## Task 1: Graph concepts and state design

LangGraph represents a workflow as a graph. A **StateGraph** defines the graph, nodes are functions that do work, edges define the normal transitions, and conditional edges choose the next node from the current state. The shared State object carries information from one node to the next.

For this notebook, the workflow is a product research assistant. It plans the request, retrieves product data, generates a recommendation, critiques the draft, loops back for revision when needed, then pauses for human approval before the final action.

### Graph before coding

```mermaid
graph TD
    START --> plan
    plan --> retrieve
    retrieve --> generate
    generate --> critique
    critique -->|quality < threshold and retries remain| generate
    critique -->|acceptable or retries exhausted| approval
    approval -->|approve| finish
    approval -->|reject| finish
    finish --> END
```

The cycle is the critique -> generat transition. retry_count prevents the graph from looping forever.

In [27]:
!pip -q install -U "langgraph>=1.0,<2" "langchain-google-genai==4.4.0" pandas pydantic typing_extensions

In [28]:
import os
import getpass
import pandas as pd
from typing import TypedDict, List, Dict, Any

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import interrupt, Command

if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Gemini API Key: ")

MODEL_NAME = "gemini-3.6-flash"
llm = ChatGoogleGenerativeAI(model=MODEL_NAME)

class WorkflowState(TypedDict, total=False):
    user_request: str
    plan: str
    retrieved: List[Dict[str, Any]]
    draft: str
    critique: str
    quality_score: int
    retry_count: int
    max_retries: int
    approval: str
    final_answer: str
    logs: List[str]

## Task 2: Build a linear graph

The first version is intentionally linear: plan -> retrieve -> generate -> format. The retrieve node reuses the product-price idea from Day 2 by reading local product data. Each node returns only the state fields it changes.

In [29]:
products = pd.DataFrame([
    {"product_name": "Wireless Mouse", "price": 19.99, "in_stock": True},
    {"product_name": "Mechanical Keyboard", "price": 69.99, "in_stock": True},
    {"product_name": "USB-C Hub", "price": 29.99, "in_stock": True},
])
products.to_csv("products.csv", index=False)

@tool
def get_product_price(product_name: str) -> str:
    """Look up a product price and stock status from the local products CSV."""
    df = pd.read_csv("products.csv")
    match = df[df["product_name"].str.lower() == product_name.lower()]
    if match.empty:
        return f"{product_name}: not found"
    row = match.iloc[0]
    stock = "in stock" if bool(row["in_stock"]) else "out of stock"
    return f"{row['product_name']}: ${row['price']:.2f} ({stock})"


def plan_node(state: WorkflowState):
    logs = state.get("logs", []) + ["plan"]
    return {"plan": f"Compare the products relevant to: {state['user_request']}", "logs": logs}


def retrieve_node(state: WorkflowState):
    logs = state.get("logs", []) + ["retrieve"]
    names = ["Wireless Mouse", "Mechanical Keyboard", "USB-C Hub"]
    data = []
    for name in names:
        result = get_product_price.invoke({"product_name": name})
        data.append({"product": name, "result": result})
    return {"retrieved": data, "logs": logs}


def generate_node(state: WorkflowState):
    logs = state.get("logs", []) + [f"generate (pass {state.get('retry_count', 0) + 1})"]
    context = "\n".join(item["result"] for item in state["retrieved"])
    prompt = f"""You are a product recommendation assistant.
User request: {state['user_request']}
Product data:
{context}
Write a short recommendation for a budget-conscious client. Mention at least two products and their prices. Do not invent prices."""
    response = llm.invoke(prompt)
    return {"draft": response.content if isinstance(response.content, str) else str(response.content), "logs": logs}


def format_node(state: WorkflowState):
    logs = state.get("logs", []) + ["format"]
    return {"final_answer": state["draft"], "logs": logs}

linear_builder = StateGraph(WorkflowState)
linear_builder.add_node("plan", plan_node)
linear_builder.add_node("retrieve", retrieve_node)
linear_builder.add_node("generate", generate_node)
linear_builder.add_node("format", format_node)
linear_builder.add_edge(START, "plan")
linear_builder.add_edge("plan", "retrieve")
linear_builder.add_edge("retrieve", "generate")
linear_builder.add_edge("generate", "format")
linear_builder.add_edge("format", END)
linear_graph = linear_builder.compile()

In [30]:
sample_input = {
    "user_request": "Compare the available products and recommend an affordable option.",
    "retry_count": 0,
    "max_retries": 2,
    "logs": [],
}

for event in linear_graph.stream(sample_input, stream_mode="updates"):
    print(event)

{'plan': {'plan': 'Compare the products relevant to: Compare the available products and recommend an affordable option.', 'logs': ['plan']}}
{'retrieve': {'retrieved': [{'product': 'Wireless Mouse', 'result': 'Wireless Mouse: $19.99 (in stock)'}, {'product': 'Mechanical Keyboard', 'result': 'Mechanical Keyboard: $69.99 (in stock)'}, {'product': 'USB-C Hub', 'result': 'USB-C Hub: $29.99 (in stock)'}], 'logs': ['plan', 'retrieve']}}
{'generate': {'draft': "[{'type': 'text', 'text': 'If you are looking for the most budget-friendly option, I highly recommend the **Wireless Mouse** at **$19.99**. It is the most affordable product in the lineup, making it a great choice for daily use without a high price tag. \\n\\nBy comparison, the **USB-C Hub** costs **$29.99**, offering versatile connectivity for a reasonable price, while the **Mechanical Keyboard** sits at the higher end at **$69.99**. \\n\\nFor a budget-conscious purchase, the **Wireless Mouse ($19.99)** gives you the best overall valu

The printed updates show the state changing after each node. LangGraph does not require every node to return the whole state; a node can return just the fields it updates.

## Task 3: Conditional edges and a self-correction cycle

The full graph adds a critique node after generation. The critique checks whether the draft contains the required product information and assigns a simple quality score. If the score is below the threshold and retries remain, the conditional edge routes back to generate. Otherwise it moves to human approval.

A loop like this is awkward to express cleanly with a basic AgentExecutor because the workflow itself needs explicit state, named transitions, and a controlled retry route. LangGraph makes the loop a first-class graph edge, so the retry condition and maximum retry count are visible in the workflow.

In [31]:
QUALITY_THRESHOLD = 80

def critique_node(state: WorkflowState):
    attempt = state.get("retry_count", 0) + 1
    draft = state.get("draft", "")
    required = ["Wireless Mouse", "Mechanical Keyboard"]
    score = 100
    notes = []
    for item in required:
        if item.lower() not in draft.lower():
            score -= 25
            notes.append(f"Missing {item}")
    if "$" not in draft:
        score -= 20
        notes.append("No price shown")
    if len(draft.strip()) < 80:
        score -= 10
        notes.append("Draft is too short")
    if not notes:
        notes.append("Draft contains the required product details")
    logs = state.get("logs", []) + [f"critique: score={score}, retry={attempt}"]
    return {"quality_score": max(score, 0), "critique": "; ".join(notes), "retry_count": attempt, "logs": logs}


def route_after_critique(state: WorkflowState):
    if state["quality_score"] < QUALITY_THRESHOLD and state["retry_count"] < state.get("max_retries", 2):
        return "generate"
    return "approval"


def approval_node(state: WorkflowState):
    decision = interrupt({
        "message": "Human approval required before the final action.",
        "draft": state.get("draft", ""),
        "quality_score": state.get("quality_score", 0),
        "options": ["approve", "reject"],
    })
    decision = str(decision).strip().lower()
    logs = state.get("logs", []) + [f"human approval: {decision}"]
    return {"approval": decision, "logs": logs}


def finish_node(state: WorkflowState):
    if state.get("approval") == "approve":
        final = "Approved: " + state["draft"]
    else:
        final = "Rejected by human reviewer. No final action was taken."
    logs = state.get("logs", []) + ["finish"]
    return {"final_answer": final, "logs": logs}

builder = StateGraph(WorkflowState)
builder.add_node("plan", plan_node)
builder.add_node("retrieve", retrieve_node)
builder.add_node("generate", generate_node)
builder.add_node("critique", critique_node)
builder.add_node("approval", approval_node)
builder.add_node("finish", finish_node)

builder.add_edge(START, "plan")
builder.add_edge("plan", "retrieve")
builder.add_edge("retrieve", "generate")
builder.add_edge("generate", "critique")
builder.add_conditional_edges("critique", route_after_critique, {"generate": "generate", "approval": "approval"})
builder.add_edge("approval", "finish")
builder.add_edge("finish", END)

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

In [32]:
# Runs until the human approval interrupt
config = {"configurable": {"thread_id": "day3-demo-1"}}
initial = {
    "user_request": "Compare the products and recommend an affordable option.",
    "retry_count": 0,
    "max_retries": 2,
    "logs": [],
}

result = graph.invoke(initial, config)
print("State at interrupt:")
print(graph.get_state(config).values)
print("Next:", graph.get_state(config).next)

State at interrupt:
{'user_request': 'Compare the products and recommend an affordable option.', 'plan': 'Compare the products relevant to: Compare the products and recommend an affordable option.', 'retrieved': [{'product': 'Wireless Mouse', 'result': 'Wireless Mouse: $19.99 (in stock)'}, {'product': 'Mechanical Keyboard', 'result': 'Mechanical Keyboard: $69.99 (in stock)'}, {'product': 'USB-C Hub', 'result': 'USB-C Hub: $29.99 (in stock)'}], 'draft': "[{'type': 'text', 'text': 'For a budget-conscious client, I highly recommend the **Wireless Mouse** at **$19.99**. It is the most affordable item available and provides great daily functionality at a very low cost. \\n\\nIf you also need expanded device connectivity, the **USB-C Hub** is another excellent low-cost choice at **$29.99**. Both options are significantly more budget-friendly than the Mechanical Keyboard ($69.99), making the Wireless Mouse your best overall pick for keeping expenses to a minimum.', 'extras': {'signature': 'Ev

In [33]:
# Inspect the loop/approval log
state = graph.get_state(config).values
for entry in state.get("logs", []):
    print(entry)
print("Critique:", state.get("critique"))
print("Quality score:", state.get("quality_score"))
print("Retry count:", state.get("retry_count"))

plan
retrieve
generate (pass 1)
critique: score=100, retry=1
Critique: Draft contains the required product details
Quality score: 100
Retry count: 1


## Task 4: Human-in-the-loop and interrupts

The approval node calls interrupt() before the final action. The graph pauses and its state is checkpointed. A human can inspect the proposed action and then resume the same thread with a Command containing either approve or reject.

In a real product, human approval is appropriate for high-impact, irreversible, expensive, or externally visible actions. Full autonomy is more reasonable for low-risk actions that can be undone and have clear validation rules.

In [34]:
# Simulates a human rejection, then resumes the paused graph
rejected = graph.invoke(Command(resume="reject"), config)
print(rejected)
print("Final state:", graph.get_state(config).values["final_answer"])

{'user_request': 'Compare the products and recommend an affordable option.', 'plan': 'Compare the products relevant to: Compare the products and recommend an affordable option.', 'retrieved': [{'product': 'Wireless Mouse', 'result': 'Wireless Mouse: $19.99 (in stock)'}, {'product': 'Mechanical Keyboard', 'result': 'Mechanical Keyboard: $69.99 (in stock)'}, {'product': 'USB-C Hub', 'result': 'USB-C Hub: $29.99 (in stock)'}], 'draft': "[{'type': 'text', 'text': 'For a budget-conscious client, I highly recommend the **Wireless Mouse** at **$19.99**. It is the most affordable item available and provides great daily functionality at a very low cost. \\n\\nIf you also need expanded device connectivity, the **USB-C Hub** is another excellent low-cost choice at **$29.99**. Both options are significantly more budget-friendly than the Mechanical Keyboard ($69.99), making the Wireless Mouse your best overall pick for keeping expenses to a minimum.', 'extras': {'signature': 'EvMUCvAUARFNMg+VdjQLy7

In [35]:
# A separate thread demonstrates approval
approve_config = {"configurable": {"thread_id": "day3-demo-approved"}}
graph.invoke(initial, approve_config)
approved = graph.invoke(Command(resume="approve"), approve_config)
print("Final approved result:")
print(graph.get_state(approve_config).values["final_answer"])

Final approved result:
Approved: [{'type': 'text', 'text': 'For a budget-conscious client, I highly recommend the **Wireless Mouse** as the top affordable option at just **$19.99**. \n\nWhen compared to the **USB-C Hub** at **$29.99** and the **Mechanical Keyboard** at **$69.99**, the Wireless Mouse provides the best entry-level value to keep costs minimal. However, if extra connectivity is a priority, the USB-C Hub is also a great budget-friendly alternative under $30.', 'extras': {'signature': 'EsAQCr0QARFNMg8Ugc9GBw6BSTb8HShxC46eO/2xsDMm9cF3fUnRTBUUF25j6rik8BaPlp5cL9JrkkWTeKSftm+a/r9ZgXvT2anm1H6aWLfx8nPca+trhcwpTEvveQz60rMZ43oE7FgWJ2IF2AlRbZFQC04zd4ykxCNkAzQTB3nB+S+GcD7477//E4UfFMZBJnghhW6pBuPiD2LNEIAnLttIpNAYa3JbP3yPxCd/i9Kk8h04NLEo43MD5Ba8smVIfbuaQLmuh5pvaszAEE2iKc6B9ivp7BynZNL2gUvQnNTs3eXVuZ4fL9cbGQOS8ZOBjqoMg9TGmOcUah2wioDMW7PxxnnoPkHuCC5l355+h+/kvfFDGjed/aCj6loD4DHtwpBkS3uWYiskX+FNmdeIQOB/kNFnkag6PLoGNSN17fyGkYxZB24B1iUC82ccQFhEkVnSUlmcMf0XTub/fSsn4OOTriKlFi8vWZvfMrnYr+2JC/D7PD

## Task 5: Persistence and debugging

The graph was compiled with InMemorySaver, so each thread has checkpointed state. The thread_id identifies which saved execution should be resumed. get_state() returns the latest checkpoint, while get_state_history() exposes the sequence of saved states for inspection and debugging.

In [36]:
# Shows persisted checkpoints for the approved thread
history = list(graph.get_state_history(approve_config))
print(f"Saved checkpoints: {len(history)}")

for i, snapshot in enumerate(reversed(history), start=1):
    print(f"Checkpoint {i}")
    print("Next:", snapshot.next)
    print("Keys:", list(snapshot.values.keys()))
    print("Retry count:", snapshot.values.get("retry_count"))
    print("Quality:", snapshot.values.get("quality_score"))

Saved checkpoints: 8
Checkpoint 1
Next: ('__start__',)
Keys: []
Retry count: None
Quality: None
Checkpoint 2
Next: ('plan',)
Keys: ['user_request', 'retry_count', 'max_retries', 'logs']
Retry count: 0
Quality: None
Checkpoint 3
Next: ('retrieve',)
Keys: ['user_request', 'plan', 'retry_count', 'max_retries', 'logs']
Retry count: 0
Quality: None
Checkpoint 4
Next: ('generate',)
Keys: ['user_request', 'plan', 'retrieved', 'retry_count', 'max_retries', 'logs']
Retry count: 0
Quality: None
Checkpoint 5
Next: ('critique',)
Keys: ['user_request', 'plan', 'retrieved', 'draft', 'retry_count', 'max_retries', 'logs']
Retry count: 0
Quality: None
Checkpoint 6
Next: ('approval',)
Keys: ['user_request', 'plan', 'retrieved', 'draft', 'critique', 'quality_score', 'retry_count', 'max_retries', 'logs']
Retry count: 1
Quality: 100
Checkpoint 7
Next: ('finish',)
Keys: ['user_request', 'plan', 'retrieved', 'draft', 'critique', 'quality_score', 'retry_count', 'max_retries', 'approval', 'logs']
Retry count: 

In [37]:
# Manual snapshot for debugging / replay-style inspection
latest = graph.get_state(approve_config)
snapshot = dict(latest.values)
print("Saved snapshot for debugging:")
print(snapshot)

Saved snapshot for debugging:
{'user_request': 'Compare the products and recommend an affordable option.', 'plan': 'Compare the products relevant to: Compare the products and recommend an affordable option.', 'retrieved': [{'product': 'Wireless Mouse', 'result': 'Wireless Mouse: $19.99 (in stock)'}, {'product': 'Mechanical Keyboard', 'result': 'Mechanical Keyboard: $69.99 (in stock)'}, {'product': 'USB-C Hub', 'result': 'USB-C Hub: $29.99 (in stock)'}], 'draft': "[{'type': 'text', 'text': 'For a budget-conscious client, I highly recommend the **Wireless Mouse** as the top affordable option at just **$19.99**. \\n\\nWhen compared to the **USB-C Hub** at **$29.99** and the **Mechanical Keyboard** at **$69.99**, the Wireless Mouse provides the best entry-level value to keep costs minimal. However, if extra connectivity is a priority, the USB-C Hub is also a great budget-friendly alternative under $30.', 'extras': {'signature': 'EsAQCr0QARFNMg8Ugc9GBw6BSTb8HShxC46eO/2xsDMm9cF3fUnRTBUUF25j6

### LangChain AgentExecutor vs LangGraph

AgentExecutor is useful when the main problem is straightforward tool calling: give the model tools, let it decide what to call, and let the executor manage the loop. LangGraph is the better choice when the application needs explicit branches, cycles, durable state, human approval, checkpoints, or precise control over what happens next.

In short, AgentExecutor is convenient for a relatively simple agent loop; LangGraph is better when the workflow itself is part of the product logic.

## Final workflow diagram

```mermaid
graph TD
    START --> plan
    plan --> retrieve
    retrieve --> generate
    generate --> critique
    critique -->|score below 80 and retries remain| generate
    critique -->|score acceptable or retries exhausted| approval
    approval -->|approve| finish
    approval -->|reject| finish
    finish --> END
